# VoxelMorph 3D — Inference

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from network import VoxelMorphUNet
from spatial_transform import SpatialTransformer
from dataset import OASISDataset
from utils import dice_similarity

In [ ]:
MODEL_PATH = "../trained_models/<run_name>/best_model.pth"
DATA_PATH  = "../data/"   # root dir with train/ and val/ subfolders
N_SAMPLES  = 4

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VoxelMorphUNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

stn = SpatialTransformer().to(device)

print(f"Loaded model from: {MODEL_PATH}")
print(f"Running on: {device}")

In [ ]:
val_dataset = OASISDataset(DATA_PATH, split="val")

indices = random.sample(range(len(val_dataset)), N_SAMPLES)

samples = []
with torch.no_grad():
    for idx in indices:
        fixed, moving, fixed_seg, moving_seg = val_dataset[idx]
        fixed  = fixed.unsqueeze(0).to(device)   # [1, 1, D, H, W]
        moving = moving.unsqueeze(0).to(device)

        flow  = model(torch.cat([fixed, moving], dim=1))
        moved = stn(moving, flow)

        samples.append((
            fixed.squeeze().cpu().numpy(),    # [D, H, W]
            moving.squeeze().cpu().numpy(),
            moved.squeeze().cpu().numpy(),
        ))

print(f"Registered {N_SAMPLES} pairs.")

In [ ]:
# Dice similarity over the full val set
model.eval()
total_dice = 0.0

with torch.no_grad():
    for idx in range(len(val_dataset)):
        fixed, moving, _, _ = val_dataset[idx]
        fixed  = fixed.unsqueeze(0).to(device)
        moving = moving.unsqueeze(0).to(device)

        flow  = model(torch.cat([fixed, moving], dim=1))
        moved = stn(moving, flow)

        total_dice += dice_similarity(fixed, moved).item()

mean_dice = total_dice / len(val_dataset)
print(f"Mean Dice similarity over val set: {mean_dice:.4f}")

## Visualizations

In [ ]:
def norm(vol):
    lo, hi = vol.min(), vol.max()
    return (vol - lo) / (hi - lo + 1e-6)

# Show the middle axial slice (along depth axis) for each sample
fig, axes = plt.subplots(N_SAMPLES, 4, figsize=(12, 3 * N_SAMPLES))
col_titles = ["Fixed", "Moving", "Warped", "Overlay (fixed=green, warped=magenta)"]

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=10)

for row, (fixed, moving, moved) in enumerate(samples):
    mid = fixed.shape[0] // 2   # middle slice along depth

    f  = norm(fixed[mid])
    mo = norm(moving[mid])
    wd = norm(moved[mid])

    H, W = f.shape
    overlay = np.zeros((H, W, 3))
    overlay[:, :, 1] = f   # green  = fixed
    overlay[:, :, 0] = wd  # red  ┐
    overlay[:, :, 2] = wd  # blue ┘ = warped → magenta

    axes[row, 0].imshow(f,       cmap="gray", vmin=0, vmax=1)
    axes[row, 1].imshow(mo,      cmap="gray", vmin=0, vmax=1)
    axes[row, 2].imshow(wd,      cmap="gray", vmin=0, vmax=1)
    axes[row, 3].imshow(overlay, vmin=0, vmax=1)

    for ax in axes[row]:
        ax.axis("off")

plt.suptitle("VoxelMorph 3D Registration Results (middle axial slice)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()